# Lean Statement Diversity Dataset — Perturbation Pipeline
Rule-based transforms — no API key required. Produces `(anchor, variant, transformation_type)` triples.

In [6]:
import re, json, math, subprocess, tempfile, os
import pandas as pd
from pathlib import Path
from huggingface_hub import HfFileSystem
from itertools import permutations
from enum import Enum
from typing import Optional
import random

fs = HfFileSystem()
indices = [132, 46, 24, 102, 85, 21, 13, 31, 20, 45]

OUTPUT_FILE = Path("perturbation_pairs.jsonl")

In [7]:
df = pd.read_json(
    "hf://datasets/FrenzyMath/mathlib_informal_v4.19.0/data.jsonl",
    lines=True,
)


df = df.iloc[indices]
# df = df[0:3]

df = df[['signature', 'type']]
df.head()

,signature,type
132,[Semigroup R] [h : PreValuationRing R] (x y :...,∀ {R : Type u_1} [inst : Semigroup R] [h : Pre...
46,": (seq q p).support = ⋃ f ∈ q.support, f '' p...",∀ {α : Type u_1} {β : Type u_2} (q : PMF (α → ...
24,(F : C ⥤ C) (X : C) : (λ_ F).hom.app X = 𝟙 _,∀ (C : Type u) [inst : CategoryTheory.Category...
102,(c₁ : Cocone F₁) (c₂ : Cocone F₂) (h : c₁.pt ...,(C : Type u₁) →\n [inst : CategoryTheory.Cate...
85,: Type,Type


In [8]:
# ── Lean runner ───────────────────────────────────────────────────────────────
PROJECT_DIR = os.path.abspath(".")
_TMP_LEAN = os.path.join(PROJECT_DIR, "_tmp_nb.lean")

def _run_lean(code: str) -> str:
    try:
        with open(_TMP_LEAN, "w") as f:
            f.write(code)
        result = subprocess.run(
            ["lake", "env", "lean", _TMP_LEAN],
            cwd=PROJECT_DIR,
            capture_output=True,
            text=True,
            timeout=300,
        )
        return result.stdout + result.stderr
    finally:
        if os.path.exists(_TMP_LEAN):
            os.remove(_TMP_LEAN)

def compile_lean(variant_sig: str, variant_type: str) -> bool:
    output = _run_lean(f"import Mathlib\n\nexample : {variant_type} := by sorry\n")
    return "error:" not in output


In [9]:
# ── is_true ───────────────────────────────────────────────────────────────────
_PROPAGATION_RULES = {
    "negation":                  {"true": "false",   "false": "true",    "unknown": "unknown"},
    "contrapositive":            {"true": "true",    "false": "false",   "unknown": "unknown"},
    "converse":                  {"true": "unknown", "false": "unknown", "unknown": "unknown"},
    "specialization":            {"true": "true",    "false": "unknown", "unknown": "unknown"},
    "generalization":            {"true": "unknown", "false": "false",   "unknown": "unknown"},
    "bound_perturbation_looser": {"true": "true",    "false": "unknown", "unknown": "unknown"},
    "bound_perturbation_tighter":{"true": "unknown", "false": "false",   "unknown": "unknown"},
}

def is_true(perturbation_path: list[str], variant_sig: str, variant_type: str) -> str:
    """
    Propagate truth symbolically through the perturbation path.
    Anchors are always 'true' (they come from Mathlib — all proven theorems).
    Returns one of: "true", "false", "unknown"
    """
    current = "true"
    for perturbation in perturbation_path:
        rules = _PROPAGATION_RULES.get(perturbation)
        current = rules[current] if rules else "unknown"
    return current

In [10]:
# ── Perturbation functions ────────────────────────────────────────────────────
def negate_theorem(sig: str, type_str: str) -> tuple[str, str] | None:
    output = _run_lean(
        f"import Negate\n\nexample : {type_str} := by\n  negate_state\n  extract_goal\n  sorry\n"
    )
    m = re.search(r"^theorem .*extracted.*$", output, re.MULTILINE)
    if m is None:
        return None
    full_statement = m.group(0)
    # Strip ":= sorry" tail
    without_proof = full_statement.rsplit(":= sorry", 1)[0].strip()
    # Split "theorem foo.extracted_1_1 : <type>" into sig and type
    parts = without_proof.split(" : ", 1)
    if len(parts) != 2:
        return None
    variant_sig, variant_type = parts
    return variant_sig.strip(), variant_type.strip()

# Just for testing, not the actual function
def converse(sig: str, type_: str) -> tuple[str, str]:
    return "conversed: " + sig, "conversed: " + type_

# Just for testing, not the actual function
def generalize(sig: str, type_: str) -> tuple[str, str]:
    return "generalized: " + sig, "generalized: " + type_

# Can edit later
TRANSFORMS = {
    "negate":     negate_theorem,
    "converse":   converse,
    "generalize": generalize,
}

In [11]:
# ── apply_perturbation_chains ─────────────────────────────────────────────────
def apply_perturbation_chains(
    df: pd.DataFrame,
    transforms: dict,
    n_permutations: int = 6,
    random_seed: int = 42,
) -> pd.DataFrame:
    rng = random.Random(random_seed)
    transform_names = list(transforms.keys())
    all_permutations = list(permutations(transform_names))

    results = []

    for _, row in df.iterrows():
        anchor_sig  = row["signature"]
        anchor_type = row["type"]

        k = min(n_permutations, len(all_permutations))
        sampled_permutations = rng.sample(all_permutations, k)

        for perm in sampled_permutations:
            current_sig  = anchor_sig
            current_type = anchor_type
            applied_so_far = []

            for transform_name in perm:
                fn = transforms[transform_name]

                result = fn(current_sig, current_type)
                if result is None:
                    break

                variant_sig, variant_type = result
                if (variant_sig.strip() == current_sig.strip()
                        and variant_type.strip() == current_type.strip()):
                    break

                if not compile_lean(variant_sig, variant_type):
                    break

                # Compiled — save this intermediate as a datapoint
                applied_so_far.append(transform_name)
                results.append({
                    **{k: v for k, v in row.items() if k not in ("signature", "type")},
                    "anchor_signature":      anchor_sig,
                    "anchor_type":           anchor_type,
                    "variant_signature":     variant_sig,
                    "variant_type":          variant_type,
                    "perturbations_applied": list(applied_so_far),
                    "chain_depth":           len(applied_so_far),
                    "is_true":               is_true(list(applied_so_far), variant_sig, variant_type),
                })

                current_sig  = variant_sig
                current_type = variant_type

    result_df = pd.DataFrame(results)
    print(f"Generated {len(result_df):,} compiled variants from {len(df):,} anchors")
    if len(result_df):
        print(result_df["chain_depth"].value_counts().sort_index().to_string())
    return result_df
